# Short-Volshock Reward / Checkpoint Screen v1

This notebook turns the current bottleneck into a repeatable reward/checkpoint pre-screen.

## Question

Can the latest short-volshock BC/checkpoint candidate justify longer PPO or promotion?

## Baselines and role

- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `short_volshock_bc_checkpoint_alignment_v1` and `short_volshock_drawdown_drift_feature_audit_v1`
- Role: `diagnose`
- Changed layer: `reward`, `model_selection`, `src_capability`
- Target: walk-forward `+30pp` relative return and `<=15%` MaxDD
- External basis: Qlib/FinRL-style walk-forward selection, risk-sensitive RL, constrained policy optimization, and checkpoint/objective screening from prior card sources.

## Historical duplicate check

- Closest success: short-volshock BC target fits the de-risk labels with high short-volshock slice accuracy.
- Closest failure: warm-start replay still misses walk-forward gates because one fold is drawdown control and the other is joint tradeoff.
- Difference here: this notebook does not train or retune thresholds; it screens the reward/checkpoint objective terms required before another PPO attempt.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/Users/rziiiii/.openclaw/workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.reward_checkpoint_screen import (
    render_reward_checkpoint_screen_review,
    score_reward_checkpoint_candidates,
    summarize_reward_checkpoint_screen,
)

EXPERIMENT = "short_volshock_reward_checkpoint_screen_v1"
CARD_DIR = ROOT / "runtime" / "reward" / "card" / EXPERIMENT
SOURCE_BC = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_bc_checkpoint_alignment_v1"
SOURCE_AUDIT = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_drawdown_drift_feature_audit_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0

print(f"card_dir={CARD_DIR.relative_to(ROOT)}")
print(f"source_bc={SOURCE_BC.relative_to(ROOT)}")
print(f"source_audit={SOURCE_AUDIT.relative_to(ROOT)}")


card_dir=runtime/reward/card/short_volshock_reward_checkpoint_screen_v1
source_bc=runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1
source_audit=runtime/model_selection/card/short_volshock_drawdown_drift_feature_audit_v1


## Inputs

The screen consumes the latest gate-gap attribution, BC slice scores, poor-window clusters, and feature-group audit. Feature groups are mechanism hypotheses only; they are not promoted as live factors here.


In [2]:
gate_gaps = pd.read_csv(SOURCE_AUDIT / "gate_gap_attribution.csv")
slice_scores = pd.read_csv(SOURCE_BC / "bc_candidate_screen.csv")
poor_window_clusters = pd.read_csv(SOURCE_AUDIT / "poor_window_clusters.csv")
feature_groups = pd.read_csv(SOURCE_AUDIT / "feature_group_summary.csv")
recipe = json.loads((SOURCE_AUDIT / "reward_checkpoint_recipe.json").read_text(encoding="utf-8"))
bc_summary = json.loads((SOURCE_BC / "summary.json").read_text(encoding="utf-8"))
audit_summary = json.loads((SOURCE_AUDIT / "summary.json").read_text(encoding="utf-8"))

input_overview = pd.DataFrame(
    [
        {"input": "gate_gaps", "rows": len(gate_gaps), "path": str((SOURCE_AUDIT / "gate_gap_attribution.csv").relative_to(ROOT))},
        {"input": "slice_scores", "rows": len(slice_scores), "path": str((SOURCE_BC / "bc_candidate_screen.csv").relative_to(ROOT))},
        {"input": "poor_window_clusters", "rows": len(poor_window_clusters), "path": str((SOURCE_AUDIT / "poor_window_clusters.csv").relative_to(ROOT))},
        {"input": "feature_groups", "rows": len(feature_groups), "path": str((SOURCE_AUDIT / "feature_group_summary.csv").relative_to(ROOT))},
    ]
)
input_overview


,input,rows,path
0,gate_gaps,6,runtime/model_selection/card/short_volshock_dr...
1,slice_scores,3,runtime/model_selection/card/short_volshock_bc...
2,poor_window_clusters,4,runtime/model_selection/card/short_volshock_dr...
3,feature_groups,7,runtime/model_selection/card/short_volshock_dr...


## Screen

A candidate is not eligible for PPO continuation unless every walk-forward window clears the fixed target gate. The ranking loss combines gate gaps, drift vs. the base reference, and short-volshock slice coverage where applicable.


In [3]:
scores = score_reward_checkpoint_candidates(
    gate_gaps,
    slice_scores=slice_scores,
    poor_window_clusters=poor_window_clusters,
    feature_groups=feature_groups,
    reference_candidate="base_multi_obj_plain",
)
summary = summarize_reward_checkpoint_screen(scores)
review_md = render_reward_checkpoint_screen_review(scores, summary)

score_columns = [
    "candidate",
    "full_gate",
    "reward_checkpoint_loss",
    "gate_gap_loss",
    "relative_drift_loss",
    "max_dd_drift_loss",
    "short_volshock_slice_loss",
    "gate_pass_count",
    "window_count",
    "min_relative_return",
    "max_max_dd",
    "failure_mix",
    "required_objective_terms",
]
scores[score_columns]


,candidate,full_gate,reward_checkpoint_loss,gate_gap_loss,relative_drift_loss,max_dd_drift_loss,short_volshock_slice_loss,gate_pass_count,window_count,min_relative_return,max_max_dd,failure_mix,required_objective_terms
0,short_volshock_plain,False,121.716955,114.531866,28.740358,0.0,0.0,0,2,5.018899,22.261989,"drawdown_control:1,joint_tradeoff:1","[""checkpoint_objective_loss"", ""drawdown_drift_..."
1,short_volshock_weighted_obj025_cb099,False,124.870064,118.636013,24.936205,0.0,0.0,0,2,8.823051,24.745906,"drawdown_control:1,selection_bias:1","[""checkpoint_objective_loss"", ""drawdown_drift_..."
2,base_multi_obj_plain,False,272.392601,272.392601,0.000000,0.0,0.0,0,2,33.759257,36.332029,drawdown_control:2,"[""checkpoint_objective_loss"", ""drawdown_drift_..."


In [4]:
selected_candidate = str(summary["selected_candidate"])
selected_gate_gaps = gate_gaps[gate_gaps["profile"].astype(str).eq(selected_candidate)].copy()
selected_gate_gaps[[
    "profile",
    "window",
    "relative_return",
    "max_dd",
    "relative_gap",
    "max_dd_gap",
    "gate_pass",
    "failure_type",
    "primary_optimization_path",
]]


,profile,window,relative_return,max_dd,relative_gap,max_dd_gap,gate_pass,failure_type,primary_optimization_path
2,short_volshock_plain,wf_2022,67.624396,22.261989,0.000000,7.261989,False,drawdown_control,drawdown_controller_or_sequence_risk_feature
4,short_volshock_plain,wf_2023_2024,5.018899,16.693087,24.981101,1.693087,False,joint_tradeoff,walk_forward_weighted_multi_objective_selection


## Bottleneck Readout

The current failure is a joint objective conflict: the actor can learn short-volshock de-risk labels, but the resulting policy still has too much drawdown in the stress fold and too little trend participation in the later walk-forward fold.


In [5]:
top_feature_groups = feature_groups.sort_values(
    ["max_stability_score", "mean_abs_delta_z"], ascending=[False, False]
).head(5)
cluster_summary = poor_window_clusters["dominant_failure_type"].astype(str).value_counts().to_dict()

readout = {
    "selected_candidate": selected_candidate,
    "decision": summary["decision"],
    "ppo_allowed_next": summary["ppo_allowed_next"],
    "promotion_allowed": summary["promotion_allowed"],
    "selected_failure_mix": summary["selected_failure_mix"],
    "selected_min_relative_return_pp": summary["selected_min_relative_return"],
    "selected_max_max_dd_pct": summary["selected_max_max_dd"],
    "poor_window_cluster_mix": cluster_summary,
    "top_feature_groups": top_feature_groups["feature_group"].tolist(),
    "required_objective_terms": summary["required_objective_terms"],
}
readout


{'selected_candidate': 'short_volshock_plain',
 'decision': 'reward_checkpoint_screen_no_promotion',
 'ppo_allowed_next': False,
 'promotion_allowed': False,
 'selected_failure_mix': 'drawdown_control:1,joint_tradeoff:1',
 'selected_min_relative_return_pp': 5.018898654205939,
 'selected_max_max_dd_pct': 22.26198899445525,
 'poor_window_cluster_mix': {'underparticipation': 4},
 'top_feature_groups': ['trend_quality',
  'volatility_regime',
  'trend_momentum',
  'relative_strength_spread',
  'gap_risk'],
 'required_objective_terms': ['checkpoint_objective_loss',
  'drawdown_drift_penalty',
  'short_volshock_state_deleverage_consistency',
  'trend_preservation_floor']}

## Write Sidecars

The notebook is the primary conclusion. Sidecars make the result indexable and reusable by later harness checks.


In [6]:
def clean_for_json(value):
    if isinstance(value, dict):
        return {str(k): clean_for_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clean_for_json(v) for v in value]
    if isinstance(value, tuple):
        return [clean_for_json(v) for v in value]
    if pd.isna(value) if not isinstance(value, (list, dict, tuple)) else False:
        return None
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    return value

created_at = datetime.now(timezone.utc).isoformat()
external_sources = sorted(set(bc_summary.get("external_sources", [])) | set(audit_summary.get("external_sources", [])))

selection = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "role": "diagnose",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [
        "short_volshock_bc_checkpoint_alignment_v1",
        "short_volshock_drawdown_drift_feature_audit_v1",
    ],
    "changed_layer": ["reward", "model_selection", "src_capability"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "decision": summary["decision"],
    "selected_candidate": selected_candidate,
    "selected_reward_checkpoint_loss": summary["selected_reward_checkpoint_loss"],
    "selected_gate_gap_loss": summary["selected_gate_gap_loss"],
    "selected_failure_mix": summary["selected_failure_mix"],
    "selected_min_relative_return_pp": summary["selected_min_relative_return"],
    "selected_max_max_dd_pct": summary["selected_max_max_dd"],
    "required_objective_terms": summary["required_objective_terms"],
    "next_mechanism": summary["next_mechanism"],
    "promotion_allowed": False,
    "ppo_allowed_next": summary["ppo_allowed_next"],
    "training_allowed_from_this_card": False,
    "promotion_blocker": "No candidate clears both walk-forward windows; this is a reward/checkpoint pre-screen only.",
    "actor_retention": "no actor bundle is promoted or retained by this notebook",
    "external_sources": external_sources,
    "sidecars": [
        "reward_checkpoint_scores.csv",
        "reward_checkpoint_summary.json",
        "selection.json",
        "summary.json",
        "reward_checkpoint_review.md",
    ],
    "source_decisions": {
        "bc_checkpoint_decision": bc_summary.get("decision"),
        "drawdown_drift_audit_decision": audit_summary.get("decision"),
        "recipe_decision": recipe.get("decision"),
    },
}

conclusion = [
    f"Selected `{selected_candidate}`; decision `{summary['decision']}`; no promotion or actor retention.",
    "Primary bottleneck: wf_2022 remains drawdown_control while wf_2023_2024 is joint_tradeoff, so longer PPO from the current BC actor is blocked.",
    f"Selected min relative return is {summary['selected_min_relative_return']:.2f}pp and max MaxDD is {summary['selected_max_max_dd']:.2f}%, against targets {TARGET_RELATIVE_RETURN_PP:.0f}pp and {TARGET_MAX_DD_PCT:.0f}%.",
    f"Poor-window cluster mix is {cluster_summary}; top feature groups are {', '.join(top_feature_groups['feature_group'].tolist())}.",
    "Next valid path: screen/train a drawdown-drift reward/checkpoint objective with short-volshock deleveraging and trend-preservation terms before any longer PPO.",
]

summary_payload = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": summary["decision"],
    "selected_candidate": selected_candidate,
    "promotion_allowed": False,
    "ppo_allowed_next": summary["ppo_allowed_next"],
    "training_allowed_from_this_card": False,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "selected_reward_checkpoint_loss": summary["selected_reward_checkpoint_loss"],
    "selected_gate_gap_loss": summary["selected_gate_gap_loss"],
    "selected_failure_mix": summary["selected_failure_mix"],
    "selected_min_relative_return_pp": summary["selected_min_relative_return"],
    "selected_max_max_dd_pct": summary["selected_max_max_dd"],
    "required_objective_terms": summary["required_objective_terms"],
    "next_mechanism": summary["next_mechanism"],
    "bottleneck": "drawdown_drift_and_trend_preservation_joint_objective",
    "conclusion": conclusion,
    "sidecars": selection["sidecars"],
    "source_context": selection["source_decisions"],
}

scores.to_csv(CARD_DIR / "reward_checkpoint_scores.csv", index=False)
(CARD_DIR / "reward_checkpoint_summary.json").write_text(
    json.dumps(clean_for_json(summary), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD_DIR / "selection.json").write_text(
    json.dumps(clean_for_json(selection), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD_DIR / "summary.json").write_text(
    json.dumps(clean_for_json(summary_payload), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD_DIR / "reward_checkpoint_review.md").write_text(review_md, encoding="utf-8")

print("wrote sidecars:")
for name in selection["sidecars"]:
    print("-", (CARD_DIR / name).relative_to(ROOT))

wrote sidecars:
- runtime/reward/card/short_volshock_reward_checkpoint_screen_v1/reward_checkpoint_scores.csv
- runtime/reward/card/short_volshock_reward_checkpoint_screen_v1/reward_checkpoint_summary.json
- runtime/reward/card/short_volshock_reward_checkpoint_screen_v1/selection.json
- runtime/reward/card/short_volshock_reward_checkpoint_screen_v1/summary.json
- runtime/reward/card/short_volshock_reward_checkpoint_screen_v1/reward_checkpoint_review.md


In [7]:
display(Markdown(review_md))


# Reward / Checkpoint Screen

## Decision

- decision: `reward_checkpoint_screen_no_promotion`
- selected_candidate: `short_volshock_plain`
- next_mechanism: `train_drawdown_drift_reward_or_checkpoint_recipe`

## Required Objective Terms

- `checkpoint_objective_loss`
- `drawdown_drift_penalty`
- `short_volshock_state_deleverage_consistency`
- `trend_preservation_floor`

## Candidate Ranking

| candidate | full | loss | pass/windows | min rel | max dd | failures |
| --- | --- | ---: | ---: | ---: | ---: | --- |
| short_volshock_plain | False | 121.72 | 0/2 | 5.02pp | 22.26% | drawdown_control:1,joint_tradeoff:1 |
| short_volshock_weighted_obj025_cb099 | False | 124.87 | 0/2 | 8.82pp | 24.75% | drawdown_control:1,selection_bias:1 |
| base_multi_obj_plain | False | 272.39 | 0/2 | 33.76pp | 36.33% | drawdown_control:2 |

## Readout

- This screen is a pre-PPO reward/checkpoint diagnostic, not runtime promotion.
- If no full-gate candidate exists, use the required terms as the next reward/checkpoint recipe.


## Conclusion

`short_volshock_plain` remains the closest diagnostic candidate, but it is not promotable and should not be extended with longer PPO from the current actor. The bottleneck is the joint objective: reduce drawdown drift in `wf_2022` without globally suppressing trend participation in `wf_2023_2024`.

Required next objective terms:

- `drawdown_drift_penalty`
- `short_volshock_state_deleverage_consistency`
- `trend_preservation_floor`
- `checkpoint_objective_loss`

No profile, actor, runtime overlay, or live factor is promoted from this notebook.
